# Yelp User Dataset Analysis with PySpark
This notebook reads and analyzes the large-scale Yelp User dataset (`yelp_academic_dataset_user.json`, ~3.1 GB) using PySpark.

In [1]:
import os
import sys
import time

# Set environment variables for PySpark to run correctly on Windows
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_SUBMIT_OPTS'] = '-Djava.security.manager=allow'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

## 1. Initialize Spark Session
We configure Spark to run locally with all available cores and allocate 4GB of memory to the driver process since the user JSON file is around 3.1 GB. We also enable the JVM security manager to support filesystem operations on Java 18+.

In [2]:
start_time = time.time()

spark = SparkSession.builder \
    .appName("Yelp_User_Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print(f"Spark Session initialized in {time.time() - start_time:.2f} seconds.")
print(f"Spark Version: {spark.version}")

Spark Session initialized in 12.26 seconds.
Spark Version: 4.1.2


## 2. Load the Yelp User JSON Dataset
We load the line-delimited JSON dataset. Note that Spark loads JSON lazily, but it will scan the schema.

In [3]:
dataset_path = "../data/raw/yelp_academic_dataset_user.json"

print(f"Loading dataset from: {dataset_path}")
start_time = time.time()

df = spark.read.json(dataset_path)

print(f"Dataset loaded and schema inferred in {time.time() - start_time:.2f} seconds.")

Loading dataset from: ../data/raw/yelp_academic_dataset_user.json


Dataset loaded and schema inferred in 15.27 seconds.


## 3. Schema Exploration
Let's print the schema of the Yelp User dataset to understand the columns and their data types.

In [4]:
df.printSchema()

root
 |-- average_stars: double (nullable = true)
 |-- compliment_cool: long (nullable = true)
 |-- compliment_cute: long (nullable = true)
 |-- compliment_funny: long (nullable = true)
 |-- compliment_hot: long (nullable = true)
 |-- compliment_list: long (nullable = true)
 |-- compliment_more: long (nullable = true)
 |-- compliment_note: long (nullable = true)
 |-- compliment_photos: long (nullable = true)
 |-- compliment_plain: long (nullable = true)
 |-- compliment_profile: long (nullable = true)
 |-- compliment_writer: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- elite: string (nullable = true)
 |-- fans: long (nullable = true)
 |-- friends: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- yelping_since: string (nullable = true)



## 4. Preview Sample Records
Let's show a preview of the first 5 records with key columns selected.

In [5]:
preview_columns = ["user_id", "name", "review_count", "yelping_since", "fans", "average_stars"]
df.select(preview_columns).show(5, truncate=False)

+----------------------+------+------------+-------------------+----+-------------+
|user_id               |name  |review_count|yelping_since      |fans|average_stars|
+----------------------+------+------------+-------------------+----+-------------+
|qVc8ODYU5SZjKXVBgXdI7w|Walker|585         |2007-01-25 16:47:26|267 |3.91         |
|j14WgRoU_-2ZE1aw1dXrJg|Daniel|4333        |2009-01-25 04:35:42|3138|3.74         |
|2WnXYQFK0hXEoTxPtV2zvg|Steph |665         |2008-07-25 10:41:00|52  |3.32         |
|SZDeASXq7o05mMNLshsdIA|Gwen  |224         |2005-11-29 04:38:33|28  |4.27         |
|hA5lMy-EnncsH4JoR-hFGQ|Karen |79          |2007-01-05 19:40:59|1   |3.54         |
+----------------------+------+------------+-------------------+----+-------------+
only showing top 5 rows


## 5. Dataset Metrics & Summary Statistics
Let's count the total number of users and compute basic summary statistics on numeric columns like `review_count`, `fans`, and `average_stars`.

In [6]:
print("Calculating total user count...")
start_time = time.time()
total_users = df.count()
print(f"Total Users in Dataset: {total_users:,} (calculated in {time.time() - start_time:.2f} seconds)")

Calculating total user count...


Total Users in Dataset: 1,987,897 (calculated in 4.83 seconds)


In [7]:
print("Calculating summary statistics...")
start_time = time.time()
df.select("review_count", "fans", "average_stars").describe().show()
print(f"Summary statistics calculated in {time.time() - start_time:.2f} seconds.")

Calculating summary statistics...


+-------+------------------+------------------+------------------+
|summary|      review_count|              fans|     average_stars|
+-------+------------------+------------------+------------------+
|  count|           1987897|           1987897|           1987897|
|   mean|23.394409267683386|1.4657404282012598|  3.63049415035087|
| stddev| 82.56699161797889| 18.13075272385579|1.1833369995975145|
|    min|                 0|                 0|               1.0|
|    max|             17473|             12497|               5.0|
+-------+------------------+------------------+------------------+

Summary statistics calculated in 5.32 seconds.


## 6. Analysis: Top 10 Most Active Users
Let's find the top 10 users with the highest review counts.

In [8]:
print("Top 10 users by review count:")
df.select("name", "review_count", "fans", "average_stars") \
  .orderBy(desc("review_count")) \
  .show(10)

Top 10 users by review count:


+--------+------------+----+-------------+
|    name|review_count|fans|average_stars|
+--------+------------+----+-------------+
|     Fox|       17473|3493|         3.77|
|  Victor|       16978|1462|         3.35|
|   Bruce|       16567| 867|         3.67|
|   Shila|       12868| 300|         3.87|
|     Kim|        9941| 825|         3.81|
|  Nijole|        8363| 921|         3.75|
| Vincent|        8354| 362|         3.87|
|  George|        7738| 288|         3.49|
| Kenneth|        6766| 285|         3.32|
|Jennifer|        6679| 828|         3.34|
+--------+------------+----+-------------+
only showing top 10 rows


## 7. Analysis: Top 10 Users by Fans
Let's find the top 10 users with the most fans.

In [9]:
print("Top 10 users by fans count:")
df.select("name", "review_count", "fans", "average_stars") \
  .orderBy(desc("fans")) \
  .show(10)

Top 10 users by fans count:


+-------+------------+-----+-------------+
|   name|review_count| fans|average_stars|
+-------+------------+-----+-------------+
|   Mike|        1882|12497|         4.39|
|  Katie|        1825| 3642|         4.23|
|    Fox|       17473| 3493|         3.77|
|Richard|        1424| 3243|         4.18|
| Daniel|        4333| 3138|         3.74|
|Jessica|        2101| 2627|         4.22|
|  Ruggy|        2434| 2547|         3.98|
|  Megan|         949| 2451|          4.4|
|    Emi|        1926| 2424|         4.33|
|  Peter|        3193| 2388|         4.01|
+-------+------------+-----+-------------+
only showing top 10 rows


## 8. Clean up Spark Session

In [10]:
spark.stop()
print("Spark Session stopped successfully.")

Spark Session stopped successfully.
